## M2 — Variables económicas nuevas
Pensión media por municipio, precio del m² y variables de demanda refinadas.
Carga desde dataset_maestro.csv — ejecutable de forma independiente.

In [6]:
import pandas as pd
import numpy as np
import unicodedata
import warnings
warnings.filterwarnings('ignore')

RUTA_BASE    = r'C:\Users\jcrs2\Documents\Akademia\akademia-residencias'
PENSION_PATH = rf'{RUTA_BASE}\MU202501.xlsx'
M2_PATH      = rf'{RUTA_BASE}\valor_m2_municipio_.xlsx'

df      = pd.read_csv(rf'{RUTA_BASE}\dataset_maestro.csv',
                      dtype={'seccion_censal': str},
                      encoding='utf-8-sig')
df_muni = pd.read_csv(rf'{RUTA_BASE}\municipios_limpios.csv')
df_pir  = pd.read_csv(rf'{RUTA_BASE}\piramide_limpia.csv',
                      dtype={'seccion_censal': str})

print(f"Dataset maestro: {df.shape[0]:,} filas, {df.shape[1]} columnas")

Dataset maestro: 36,334 filas, 31 columnas


### Variable 1 — Pensión media por municipio (Seguridad Social, enero 2025)

In [7]:
xl = pd.ExcelFile(PENSION_PATH)
pestañas = xl.sheet_names[2:-1]

dfs = []
for pestaña in pestañas:
    try:
        df_raw = pd.read_excel(xl, sheet_name=pestaña, header=None)
        df_prov = df_raw.iloc[5:, [0, 1, 2, 4]].copy()
        df_prov.columns = ['cod_municipio', 'nombre',
                           'num_pensionistas', 'pension_media']
        df_prov = df_prov[
            pd.to_numeric(df_prov['cod_municipio'], errors='coerce').notna()
        ].copy()
        df_prov['pension_media']    = pd.to_numeric(df_prov['pension_media'],    errors='coerce')
        df_prov['num_pensionistas'] = pd.to_numeric(df_prov['num_pensionistas'], errors='coerce')
        dfs.append(df_prov)
    except Exception as e:
        print(f"Error en {pestaña}: {e}")

df_pension = pd.concat(dfs, ignore_index=True)
df_pension['cod_municipio'] = pd.to_numeric(
    df_pension['cod_municipio'], errors='coerce'
).astype('Int64')

df_pension['CPRO'] = (df_pension['cod_municipio'] // 1000).astype(int)
df_pension['CMUN'] = (df_pension['cod_municipio'] %  1000).astype(int)

print(f"Municipios con datos de pensión: {df_pension.shape[0]:,}")
print(f"Pensión media nacional: {df_pension['pension_media'].mean():,.0f}€/mes")
print(f"Rango: {df_pension['pension_media'].min():,.0f}€ — "
      f"{df_pension['pension_media'].max():,.0f}€")

Municipios con datos de pensión: 8,136
Pensión media nacional: 1,092€/mes
Rango: 580€ — 2,141€


### Cruzar pensión con dataset maestro

In [8]:
# Eliminar columnas si ya existen (para que el notebook sea re-ejecutable)
df = df.drop(columns=['pension_media', 'num_pensionistas'], errors='ignore')

df = df.merge(
    df_pension[['CPRO', 'CMUN', 'pension_media', 'num_pensionistas']],
    on=['CPRO', 'CMUN'],
    how='left'
)

print(f"Filas: {df.shape[0]:,} | Duplicados: {df.duplicated('seccion_censal').sum()}")
print(f"Secciones sin pensión: {df['pension_media'].isna().sum():,}")

for cpro, cmun, nombre in [(28, 115, 'Pozuelo'),
                            (28, 79,  'Madrid'),
                            (20, 69,  'Donostia')]:
    fila = df[(df['CPRO']==cpro) & (df['CMUN']==cmun)]
    if len(fila) > 0:
        print(f"  {nombre}: {fila['pension_media'].iloc[0]:,.0f}€/mes")

Filas: 36,334 | Duplicados: 0
Secciones sin pensión: 138
  Pozuelo: 1,853€/mes
  Madrid: 1,567€/mes
  Donostia: 1,683€/mes


### Variable 2 — Precio del m² de vivienda por municipio (Ministerio de Transportes, Q4 2025)

In [9]:
xl_m2 = pd.ExcelFile(M2_PATH)
ultima_hoja = xl_m2.sheet_names[-1]

df_m2_raw = pd.read_excel(xl_m2, sheet_name=ultima_hoja, header=None)
df_data   = df_m2_raw.iloc[17:].copy().reset_index(drop=True)

df_data.columns = range(len(df_data.columns))
df_data[1] = df_data[1].ffill()

df_m2 = pd.DataFrame({
    'provincia_m2': df_data[1].astype(str).str.strip(),
    'municipio_m2': df_data[2].astype(str).str.strip(),
    'precio_m2':    pd.to_numeric(df_data[5], errors='coerce')
})

df_m2 = df_m2[
    (df_m2['municipio_m2'].notna()) &
    (df_m2['municipio_m2'] != 'nan') &
    (df_m2['precio_m2'].notna())
].reset_index(drop=True)

print(f"Municipios con precio m²: {len(df_m2):,}")
print(f"Precio medio: {df_m2['precio_m2'].mean():,.0f} €/m²")
print(f"Rango: {df_m2['precio_m2'].min():,.0f} — {df_m2['precio_m2'].max():,.0f} €/m²")

Municipios con precio m²: 306
Precio medio: 2,102 €/m²
Rango: 635 — 6,233 €/m²


### Cruzar precio m² con dataset maestro

In [10]:
def normalizar(s):
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s

df_m2['muni_norm']  = df_m2['municipio_m2'].apply(normalizar)
df_muni['muni_norm'] = df_muni['NOMBRE'].apply(normalizar)

df_m2_merge = df_muni[['CPRO', 'CMUN', 'muni_norm']].merge(
    df_m2[['muni_norm', 'precio_m2']],
    on='muni_norm',
    how='inner'
).drop_duplicates(subset=['CPRO', 'CMUN'])

df = df.drop(columns=['precio_m2', 'precio_m2_final'], errors='ignore')

df = df.merge(
    df_m2_merge[['CPRO', 'CMUN', 'precio_m2']],
    on=['CPRO', 'CMUN'],
    how='left'
)

# Fallback provincial para municipios pequeños
precio_prov = df[df['precio_m2'].notna()].groupby('CPRO')['precio_m2'].mean()
df['precio_m2_final'] = df['precio_m2'].copy()
mask = df['precio_m2_final'].isna()
df.loc[mask, 'precio_m2_final'] = df.loc[mask, 'CPRO'].map(precio_prov)

print(f"Municipios cruzados: {len(df_m2_merge):,}")
print(f"Secciones sin precio tras fallback: {df['precio_m2_final'].isna().sum():,}")

for cpro, cmun, nombre in [(28, 115, 'Pozuelo'),
                            (28, 79,  'Madrid'),
                            (8,  19,  'Barcelona')]:
    fila = df[(df['CPRO']==cpro) & (df['CMUN']==cmun)]
    if len(fila) > 0:
        print(f"  {nombre}: {fila['precio_m2_final'].iloc[0]:,.0f} €/m²")

Municipios cruzados: 280
Secciones sin precio tras fallback: 241
  Pozuelo: 5,102 €/m²
  Madrid: 5,286 €/m²
  Barcelona: 4,501 €/m²


### Variables de demanda refinadas — tasa de sustitución y capa 80+

In [11]:
# Tasa de sustitución
df = df.drop(columns=['tasa_sustitucion'], errors='ignore')
df['tasa_sustitucion'] = (
    df['cohorte_futura'] / df['vol_75plus'].replace(0, np.nan)
)

# Capa 80+
cols_80plus = ['ambos_80_84', 'ambos_85_89', 'ambos_90_94',
               'ambos_95_99', 'ambos_100mas']

df_80 = df_pir[['seccion_censal'] + cols_80plus].copy()
df_80['vol_80plus'] = df_80[cols_80plus].sum(axis=1)
df_80['pct_80plus'] = df_80['vol_80plus'] / df_pir['total_ambos'] * 100

df = df.drop(columns=['vol_80plus', 'pct_80plus'], errors='ignore')
df = df.merge(
    df_80[['seccion_censal', 'vol_80plus', 'pct_80plus']],
    on='seccion_censal',
    how='left'
)

print(f"tasa_sustitucion — media: {df['tasa_sustitucion'].mean():.2f}")
print(f"vol_80plus       — media: {df['vol_80plus'].mean():.1f}")
print(f"pct_80plus       — media: {df['pct_80plus'].mean():.1f}%")
print(f"\nFilas: {df.shape[0]:,} | Duplicados: {df.duplicated('seccion_censal').sum()}")

tasa_sustitucion — media: 1.10
vol_80plus       — media: 78.9
pct_80plus       — media: 7.5%

Filas: 36,334 | Duplicados: 0


### Guardar dataset maestro final

In [12]:
df.to_csv(rf'{RUTA_BASE}\dataset_maestro.csv', index=False, encoding='utf-8-sig')

print("=== DATASET MAESTRO FINAL ===")
print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"\nVerificación final:")
for cpro, cmun, nombre in [(28, 115, 'Pozuelo'),
                            (46, 250, 'Valencia'),
                            (20, 69,  'Donostia')]:
    fila = df[(df['CPRO']==cpro) & (df['CMUN']==cmun)]
    if len(fila) > 0:
        f = fila.iloc[0]
        print(f"  {nombre}: pensión={f['pension_media']:,.0f}€ | "
              f"m²={f['precio_m2_final']:,.0f}€ | "
              f"tasa_sust={f['tasa_sustitucion']:.2f} | "
              f"vol_80+={f['vol_80plus']:.0f}")

=== DATASET MAESTRO FINAL ===
Filas:    36,334
Columnas: 31

Verificación final:
  Pozuelo: pensión=1,853€ | m²=5,102€ | tasa_sust=1.00 | vol_80+=163
  Valencia: pensión=1,424€ | m²=2,791€ | tasa_sust=1.41 | vol_80+=41
  Donostia: pensión=1,683€ | m²=2,672€ | tasa_sust=0.76 | vol_80+=63
